# Two-stage model (`two_stage_lasso_rf`): Lasso for the level, random forest for the flat: training and inference

The same two-stage model as `neural_network/two_stage_nn_forest.ipynb`, with a **Lasso regression instead of the neural network** as the stage-1 trend model.

* **Stage 1 (level by month):** a Lasso on the 177 temporal features of `data/features/final/temporal_scaled.csv` (stock, COE and macroeconomic features, scaled by the data pipeline) that predicts the rent level of each month.
  It is fitted on the 39 train months (2021-01..2024-03), each month weighted by its number of rows. This is exactly the Lasso on the individual rows (every flat of a month has the same features),
  so the months with more rentals count for more, as they do for the network. The rent is standardised for the fit, so the penalty `alpha` is in standard deviations of rent.
  **`alpha` is tuned on the validation rows** (section 3): the Lasso fitted on the train months is scored on the 35,158 rows of 2024-04..2025-03 and the `alpha` with the lowest validation RMSE is kept.
* **Stage 2 (the flat):** a random forest on the static features (151) that predicts the **ratio** `rent / level of its month`, weighted by `level^2` so that its squared error in ratio is the squared error in dollars.
  The forest settings are chosen on the validation rows with the true monthly level (a perfect stage 1), as in the network version.
* **Prediction:** `level of the month x ratio of the flat`. The forest never sees the month, so the growth of the same flats equals the growth of the stage-1 level by construction.

**One fit.** Both stages are fitted on the train months only and never refitted on the validation months: the test rows have no rents, so the validation rows are the only held-out rents the models can be compared on.
The same fitted model predicts the train rows (in-sample), the validation rows (held out), the 50,000 test rows (a forecast) and the rent of a fixed set of flats in every month (the flats of 2025-01..03, features only).

* **Outputs:** the model in `results/models/`, the tuning tables in `results/tables/two_stage_lasso_rf/`, and the predictions (with the identifier columns of the static features) and fixed-flat levels in `03_evaluation/predictions/`,
  in the same format as the neural networks. Scoring, the growth benchmark and the charts are in `03_evaluation/evaluation.ipynb`.

Run it from this folder. It uses the data loading and saving helpers of `02_modelling/nn_utils.py` (no network is trained here).

In [1]:
import sys
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Lasso

import os, sys
from pathlib import Path
_here = Path(os.environ.get('RENT_PROJECT_ROOT') or Path.cwd()).resolve()
sys.path.insert(0, str(next(p for p in [_here, *_here.parents] if (p / 'data' / 'config.py').exists())))      # final_report: data/config.py holds every path
from data import config as cfg
sys.path.insert(0, str(cfg.MODELLING_DIR))          # 02_modelling: nn_utils.py
import nn_utils as nu

warnings.filterwarnings('ignore')
pd.set_option('display.width', 220)
NAME = 'two_stage_lasso_rf'
nu.use_results_dir(cfg.TWO_STAGE_RESULTS_DIR)
nu.ensure_output_dirs(NAME)

## 1. Data

In [2]:
train, test, STATIC = nu.load_static()
fit, val = (train['sample_type'] == 'train').to_numpy(), (train['sample_type'] == 'val').to_numpy()
month, y = train['month'].to_numpy(), train[nu.LABEL].to_numpy()
X, X_test = train[STATIC].to_numpy(np.float32), test[STATIC].to_numpy(np.float32)
month_mean = train.groupby('month')[nu.LABEL].mean()                                 # the actual level of each month 0..50
y_val, m_val = y[val], month[val]
fixed = np.isin(month, nu.Q1_2025)                                                   # the fixed flats: the rows of 2025-01..03 (their features only, never their rents)

temporal = nu.load_temporal()                                                        # 67 months x 177 scaled features
print(f'static {len(STATIC)} features, temporal {temporal.shape[1]} features; fit rows {fit.sum():,} (months 0..{nu.TRAIN_LAST}), validation rows {val.sum():,} '
      f'(months {nu.TRAIN_LAST + 1}..{nu.VALIDATION_LAST}), test rows {len(test):,}')

static 151 features, temporal 177 features; fit rows 114,841 (months 0..38), validation rows 35,158 (months 39..50), test rows 50,000


## 2. Stage 1: the Lasso on the monthly level

`fit_lasso` fits the Lasso on the months of the given rows (each weighted by its row count) against the standardised mean rent of the month, and `month_levels` predicts the level (in dollars) of all 67 months.

In [3]:
def fit_lasso(alpha, rows):
    """Lasso of the standardised monthly mean rent on the temporal features, each month weighted by its number of rows (the same as the Lasso on the rows)."""
    counts = pd.Series(month[rows]).value_counts().sort_index()
    mean_rent = pd.Series(y[rows]).groupby(month[rows]).mean()
    y_mean, y_std = y[rows].mean(), y[rows].std()
    lasso = Lasso(alpha=alpha, max_iter=200_000).fit(temporal.loc[mean_rent.index].to_numpy(), (mean_rent.to_numpy() - y_mean) / y_std, sample_weight=counts.to_numpy(dtype=float))
    return {'lasso': lasso, 'y_mean': y_mean, 'y_std': y_std, 'alpha': alpha}

def month_levels(model):
    return pd.Series(model['lasso'].predict(temporal.to_numpy()) * model['y_std'] + model['y_mean'], index=temporal.index)

## 3. Tuning `alpha` on the validation rows

For each `alpha` of a log-spaced grid the Lasso is fitted on the train months (0..38) and the level it predicts for each validation month is scored on the validation rows: **the RMSE of the monthly level used as the prediction of every flat of the month**
(the stage-1 error in dollars on the validation rows; the forest only adds a ratio on top of it). The `alpha` with the lowest validation RMSE is kept. A larger `alpha` keeps fewer temporal features (`n_features`); at the top of the grid none is left and the level is constant.

In [4]:
ALPHAS = np.logspace(-4, 0, 25)
tuning = []
for alpha in ALPHAS:
    model = fit_lasso(alpha, fit)
    level_a = month_levels(model)
    tuning.append({'alpha': alpha, 'n_features': int((model['lasso'].coef_ != 0).sum()), 'train_rmse': nu.rmse(y[fit], level_a.loc[month[fit]]), 'validation_rmse': nu.rmse(y_val, level_a.loc[m_val]),
                   'validation_level_bias': float((level_a.loc[m_val] - y_val).mean())})
tuning = pd.DataFrame(tuning)
ALPHA = float(tuning.loc[tuning['validation_rmse'].idxmin(), 'alpha'])
tuning['selected'] = tuning['alpha'] == ALPHA
print(tuning.round(4).to_string(index=False))
print(f'\nalpha = {ALPHA:.5f}: validation RMSE {tuning.loc[tuning["selected"], "validation_rmse"].item():,.2f}, {tuning.loc[tuning["selected"], "n_features"].item()} temporal features kept')
if ALPHA in (ALPHAS[0], ALPHAS[-1]):
    print('WARNING: the best alpha is at the edge of the grid, widen it')

 alpha  n_features  train_rmse  validation_rmse  validation_level_bias  selected
0.0001          46    604.4111         661.9738               -51.8133     False
0.0001          41    604.4113         660.5157                35.4406     False
0.0002          41    604.4117         660.5947                34.3683     False
0.0003          41    604.4126         660.4646                32.6242     False
0.0005          40    604.4144         659.8975                16.4033      True
0.0007          39    604.4182         660.5260               -16.6053     False
0.0010          31    604.4261         660.6063               -22.4004     False
0.0015          31    604.4395         660.0280                -9.7518     False
0.0022          26    604.4602         660.0461               -11.4392     False
0.0032          22    604.4873         660.1285               -19.9868     False
0.0046          19    604.5388         660.1870               -27.1908     False
0.0068          14    604.62

## 4. Forest settings

Stage 1 is the Lasso at the chosen `alpha`, fitted on the train months (the only fit). The forest is fitted on the train rows: target = rent / (in-sample level of its month), weighted by `level^2`.

**Forest settings.** A small grid over the share of features tried per split and the minimum leaf size (200 trees, depth 24, 60% of the rows per tree), scored on the validation rows with the **oracle level**, the true monthly mean of each
validation month, i.e. a perfect stage 1, so that only stage 2 is judged.

In [5]:
model = fit_lasso(ALPHA, fit)
level = month_levels(model)                                                           # stage-1 level of the 67 months
level_w = level.loc[month[fit]].to_numpy()
oracle = month_mean.loc[m_val].to_numpy()
BASE_PARAMS = {'n_estimators': 200, 'max_depth': 24, 'max_samples': 0.6}

grid = []
for max_features in (0.33, 0.55, 0.85):
    for min_samples_leaf in (5, 10, 20):
        forest = RandomForestRegressor(n_jobs=-1, random_state=0, max_features=max_features, min_samples_leaf=min_samples_leaf, **BASE_PARAMS
                                       ).fit(X[fit], y[fit] / level_w, sample_weight=level_w ** 2)
        grid.append({'max_features': max_features, 'min_samples_leaf': min_samples_leaf, 'validation_rmse_oracle_level': nu.rmse(y_val, oracle * forest.predict(X[val]))})
grid = pd.DataFrame(grid).sort_values('validation_rmse_oracle_level').reset_index(drop=True)
print(grid.round(2).to_string())
RF_PARAMS = {**BASE_PARAMS, 'max_features': float(grid.loc[0, 'max_features']), 'min_samples_leaf': int(grid.loc[0, 'min_samples_leaf'])}
print('\nforest parameters:', RF_PARAMS)

   max_features  min_samples_leaf  validation_rmse_oracle_level
0          0.33                10                        468.93
1          0.55                10                        468.98
2          0.85                20                        469.34
3          0.85                10                        469.43
4          0.55                20                        469.44
5          0.33                20                        469.75
6          0.33                 5                        469.86
7          0.55                 5                        470.29
8          0.85                 5                        470.72

forest parameters: {'n_estimators': 200, 'max_depth': 24, 'max_samples': 0.6, 'max_features': 0.33, 'min_samples_leaf': 10}


## 5. The two-stage model

The forest with the chosen settings, fitted on the train rows, with the stage-1 Lasso level: `level of the month x ratio of the flat`. It predicts the train, validation and test rows; the level of the same flats in each month is
level(month) x the mean ratio of the fixed flats.

In [6]:
forest = RandomForestRegressor(n_jobs=-1, random_state=0, **RF_PARAMS).fit(X[fit], y[fit] / level_w, sample_weight=level_w ** 2)
ratio_train, ratio_val, ratio_test = forest.predict(X[fit]), forest.predict(X[val]), forest.predict(X_test)
pred_train, pred_val, pred_test = level.loc[month[fit]].to_numpy() * ratio_train, level.loc[m_val].to_numpy() * ratio_val, level.loc[test['month']].to_numpy() * ratio_test
level_fixed = level * forest.predict(X[fixed]).mean()                                  # the same flats in every month: level(m) x mean ratio of the flats

coefficients = pd.Series(model['lasso'].coef_, index=temporal.columns)
selected = coefficients[coefficients != 0].sort_values(key=np.abs, ascending=False).rename('coefficient (standardised rent per scaled feature)')
print(f'validation RMSE: stage 1 alone ${nu.rmse(y_val, level.loc[m_val]):,.1f}, two-stage ${nu.rmse(y_val, pred_val):,.1f}  (oracle level: ${grid["validation_rmse_oracle_level"].min():,.1f})')
print(f'{len(selected)} temporal features kept; in-sample ratio mean {ratio_train.mean():.3f}; test ratio mean {ratio_test.mean():.3f}')
print(selected.head(10).round(4).to_string())
print(level_fixed.loc[44:].round(0).rename(lambda m: nu.month_label(m)[0]).to_frame('level of the same flats').T.to_string())

validation RMSE: stage 1 alone $659.9, two-stage $470.7  (oracle level: $468.9)
40 temporal features kept; in-sample ratio mean 0.999; test ratio mean 1.001
MACRO_SORA_3M_PCT                               0.2635
coe_c_quota_premium                             0.0465
stk_sheng_siong_group_log_price_mean            0.0401
MACRO_UNEMPLOYMENT_SA_PCT                      -0.0386
stk_wilmar_international_log_price_mean        -0.0383
stk_singapore_post_log_price_mean              -0.0300
stk_mapletree_logistics_trust_log_price_mean   -0.0289
stk_sea_limited_log_price_mean                 -0.0216
stk_singapore_airlines_log_price_mean           0.0208
stk_singtel_log_price_mean                      0.0199
                         2024-09  2024-10  2024-11  2024-12  2025-01  2025-02  2025-03  2025-04  2025-05  2025-06  2025-07  2025-08  2025-09  2025-10  2025-11  2025-12  2026-01  2026-02  2026-03  2026-04  2026-05  2026-06  2026-07
level of the same flats   3193.0   3136.0   3155.0   3145.0   

## 6. Save the model, the tuning tables and the predictions

In [7]:
paths = [nu.MODEL_DIR / f'{NAME}.joblib']
joblib.dump({'lasso': model['lasso'], 'target_mean': model['y_mean'], 'target_std': model['y_std'], 'alpha': ALPHA, 'temporal_features': list(temporal.columns),
             'forest': forest, 'static_features': STATIC, 'rf_params': RF_PARAMS, 'target': 'rent / stage-1 level of the month, weighted by level^2'}, paths[0], compress=3)
paths += [nu.save_table(tuning, NAME, 'alpha_tuning.csv', index=False), nu.save_table(grid, NAME, 'rf_tuning.csv', index=False), nu.save_table(selected, NAME, 'selected_temporal_features.csv', index_label='feature')]
paths += [nu.save_predictions(NAME, 'train', train[fit], pred_train),      # in-sample: the 114,841 train rows
          nu.save_predictions(NAME, 'val', train[val], pred_val),          # held out: the 35,158 validation rows
          nu.save_predictions(NAME, 'test', test, pred_test),              # the 50,000 test rows
          nu.save_levels(NAME, level_fixed)]
print('saved:', *paths, sep='\n  ')

saved:
  /Users/keltonn/Desktop/Repo/school/CS5228-project/final_report/02_modelling/two_stage_models/results/models/two_stage_lasso_rf.joblib
  /Users/keltonn/Desktop/Repo/school/CS5228-project/final_report/02_modelling/two_stage_models/results/tables/two_stage_lasso_rf/alpha_tuning.csv
  /Users/keltonn/Desktop/Repo/school/CS5228-project/final_report/02_modelling/two_stage_models/results/tables/two_stage_lasso_rf/rf_tuning.csv
  /Users/keltonn/Desktop/Repo/school/CS5228-project/final_report/02_modelling/two_stage_models/results/tables/two_stage_lasso_rf/selected_temporal_features.csv
  /Users/keltonn/Desktop/Repo/school/CS5228-project/final_report/03_evaluation/predictions/train/two_stage_lasso_rf.csv
  /Users/keltonn/Desktop/Repo/school/CS5228-project/final_report/03_evaluation/predictions/val/two_stage_lasso_rf.csv
  /Users/keltonn/Desktop/Repo/school/CS5228-project/final_report/03_evaluation/predictions/test/two_stage_lasso_rf.csv
  /Users/keltonn/Desktop/Repo/school/CS5228-project